In [1]:
import pandas as pd
import numpy as np
import sys
import os

In [2]:
def parse_mixed_dates(val):
    try:
        num = float(val)
        if num > 1000:
            return pd.to_datetime(num, unit='D', origin='1899-12-30')
    except (ValueError, TypeError):
        pass
    return pd.to_datetime(val, dayfirst=True, errors='coerce')

In [3]:
def clean_time_str(val):
    if pd.isna(val):
        return val
    try:
        v = float(val)
        if 0 <= v <= 1:
            total_sec = int(v * 86400)
            return f"{total_sec // 3600:02d}:{(total_sec % 3600) // 60:02d}"
    except ValueError:
        pass
    return str(val).strip()

In [4]:
def get_time_period(hour):
    if pd.isna(hour) or hour < 0:
        return 'ไม่ระบุ'
    elif 0 <= hour < 4:
        return 'ดึก (00:00 - 03:59)'
    elif 4 <= hour < 8:
        return 'เช้ามืด (04:00 - 07:59)'
    elif 8 <= hour < 12:
        return 'เช้า (08:00 - 11:59)'
    elif 12 <= hour < 16:
        return 'บ่าย (12:00 - 15:59)'
    elif 16 <= hour < 20:
        return 'เย็น (16:00 - 19:59)'
    else:
        return 'กลางคืน (20:00 - 23:59)'

In [5]:
def calculate_severity(row):
    if row.get('ผู้เสียชีวิต', 0) >= 1:
        return 'รุนแรงมาก (เสียชีวิต)'
    elif row.get('ผู้บาดเจ็บสาหัส', 0) >= 1:
        return 'รุนแรงปานกลาง (บาดเจ็บสาหัส)'
    else:
        return 'รุนแรงน้อย (บาดเจ็บเล็กน้อย/ทรัพย์สินเสียหาย)'

In [6]:
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd

GEOJSON_PATH = "../subdistricts.geojson"

def rename_columns(df):

    rename_dict = {
        'ก.ม.': 'KM',
        'บริเวณที่เกิดเหตุ/ลักษณะทาง': 'บริเวณที่เกิดเหตุ',
        'ลักษณะการเกิดอุบัติเหตุ': 'ลักษณะการเกิดเหตุ',
        'จำนวนรถที่เกิดเหตุ(รวมคันที่1)': 'รถที่เกิดเหตุ',
        'จำนวนที่เกิดเหตุทั้งหมด(รวมคนเดินเท้า)': 'รถและคนที่เกิดเหตุ',
        'รถยนต์นั่งส่วนบุคคล/รถยนต์นั่งสาธารณะ': 'รถยนต์นั่งส่วนบุคคล',
        'รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้อ': 'รถบรรทุกไม่เกิน10ล้อ',
        'รถบรรทุกมากกว่า10ล้อ(รถพ่วง)': 'รถบรรทุกมากกว่า10ล้อ',
        'อื่นๆ': 'รถอื่นๆ',
        'จำนวนผู้เสียชีวิต': 'ผู้เสียชีวิต',
        'จำนวนผู้บาดเจ็บสาหัส': 'ผู้บาดเจ็บสาหัส',
        'จำนวนผู้บาดเจ็บเล็กน้อย': 'ผู้บาดเจ็บเล็กน้อย'
    }

    df = df.copy()

    for old_col, new_col in rename_dict.items():
        if old_col in df.columns:
            if new_col in df.columns:
                df[new_col] = df[new_col].fillna(df[old_col])
                df = df.drop(columns=[old_col])
            else:
                df = df.rename(columns={old_col: new_col})
    return df

def clean_time_str(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    try:
        parsed = pd.to_datetime(value, errors='coerce')
        if pd.notna(parsed):
            return parsed.strftime('%H:%M:%S')
    except Exception:
        pass
    return None

def parse_mixed_dates(value):
    if pd.isna(value):
        return pd.NaT
    try:
        return pd.to_datetime(
            value,
            errors='coerce',
            dayfirst=True
        )
    except Exception:
        return pd.NaT

def get_time_period(hour):

    if pd.isna(hour) or hour < 0 or hour > 23:
        return 'ไม่ระบุ'
    elif 0 <= hour < 4:
        return 'ดึก (00:00 - 03:59)'
    elif 4 <= hour < 8:
        return 'เช้ามืด (04:00 - 07:59)'
    elif 8 <= hour < 12:
        return 'เช้า (08:00 - 11:59)'
    elif 12 <= hour < 16:
        return 'บ่าย (12:00 - 15:59)'
    elif 16 <= hour < 20:
        return 'เย็น (16:00 - 19:59)'
    else:
        return 'กลางคืน (20:00 - 23:59)'
        
def calculate_severity(row):

    deaths = row.get('ผู้เสียชีวิต', 0)
    serious = row.get('ผู้บาดเจ็บสาหัส', 0)
    minor = row.get('ผู้บาดเจ็บเล็กน้อย', 0)

    deaths = 0 if pd.isna(deaths) else deaths
    serious = 0 if pd.isna(serious) else serious
    minor = 0 if pd.isna(minor) else minor

    return deaths * 25 + serious * 15 + minor * 5 + 5

def add_day_type(df):

    df = df.copy()

    df['day_type'] = np.select(
        [
            df['is_new_year'] == 1,
            df['is_songkran'] == 1,
            df['is_weekend'] == 1
        ],
        ['new_year', 'songkran', 'weekend'],
        default='normal_day'
    )
    return df

def add_tambon(df, geojson_path):

    df = df.copy()

    if not os.path.exists(geojson_path):
        raise FileNotFoundError(f"ไม่พบไฟล์ GeoJSON: {geojson_path}")

    tambon_map = gpd.read_file(geojson_path)

    if 'tam_th' not in tambon_map.columns:
        raise KeyError("ไม่พบคอลัมน์ tam_th ในไฟล์ GeoJSON")
        
    points = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(
            df['LONGITUDE'],
            df['LATITUDE']
        ),
        crs='EPSG:4326'
    )

    if tambon_map.crs != points.crs:
        tambon_map = tambon_map.to_crs(points.crs)

    result = gpd.sjoin(
        points,
        tambon_map[['tam_th', 'geometry']],
        how='left',
        predicate='intersects'
    )
    
    tambon_names = result.groupby(level=0)['tam_th'].first()
    df['tambon'] = tambon_names.reindex(df.index)
    df['tambon'] = df['tambon'].fillna('ไม่ระบุ')
    return df

def process_cleansing_pipeline(df):

    df = df.copy()
    df = rename_columns(df)
    df['LATITUDE'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
    df['LONGITUDE'] = pd.to_numeric( df['LONGITUDE'], errors='coerce')
    valid_lat = ((df['LATITUDE'] >= 5.0) & (df['LATITUDE'] <= 21.0))
    valid_long = ((df['LONGITUDE'] >= 97.0) & (df['LONGITUDE'] <= 106.0))
    df = df[valid_lat & valid_long].copy()

    numeric_cols = [
        'ผู้เสียชีวิต',
        'ผู้บาดเจ็บสาหัส',
        'ผู้บาดเจ็บเล็กน้อย',
        'รวมจำนวนผู้บาดเจ็บ'
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors='coerce'
            ).fillna(0).astype(int)

    text_cols = [
        'บริเวณที่เกิดเหตุ',
        'มูลเหตุสันนิษฐาน',
        'ลักษณะการเกิดเหตุ',
        'สภาพอากาศ',
        'จังหวัด',
        'รถคันที่1',
        'สายทาง'
    ]

    for col in text_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna('ไม่ระบุ')
                .astype(str)
                .str.strip()
            )

    if 'มูลเหตุสันนิษฐาน' in df.columns:
        df['มูลเหตุสันนิษฐาน'] = df['มูลเหตุสันนิษฐาน'].replace({
            "ขับขี่ย้อนศร": "ขับรถย้อนศร",
            "ขับชิดซ้ายมากเกินไป": "ขับรถชิดซ้ายมากเกินไป",
            "ขับรถกระชั้นชิด": "ขับรถตามกระชั้นชิด",
            "ฝ่าฝืนสัญญาณไฟ / เครื่องหมายจราจร": "ฝ่าฝืนสัญญาณไฟ/เครื่องหมายจราจร",
            "ฝ่าฝืนป้ายหยุดขณะออกจากทางร่วมทางแยก": "ฝ่าฝืนป้ายหยุดขณะออกจากทางร่วมแยก",
            "มึนเมาจากแอลกอฮอล์": "เมาสุรา",
            "รถเสียไม่แสดงเครื่องหมาย/สัญญาณตามที่กำหนด": "รถเสียไม่แสดงเครื่องหมายหรือสัญญาณไฟที่กำหนด",
            "ระบบไฟฟ้ขัดข้อง": "ระบบไฟฟ้าขัดข้อง",
            "ระบบไฟฟ้าของยานพาหนะขัดข้อง": "ระบบไฟฟ้าขัดข้อง",
            "อาการป่วย หรือการไร้ความสามารถทางประสาทหรือร่างกาย": "อาการป่วยหรือการไร้ความสามารถทางประสาทหรือร่างกาย",
            "ใช้โทรศัพท์เคลื่อนที่ขณะขับรถ": "ใช้โทรศัพท์ขณะขับรถ",
            "ไม่ให้สัญญาณชะลอ / เลี้ยว": "ไม่ให้สัญญาณชะลอ/เลี้ยว",
            "ไม่ให้สัญญาณเข้าจอด หรือออกจากที่จอด": "ไม่ให้สัญญาณเข้าจอด/ออกจากที่จอด",
            "ไม่คุ้นเคยกับการขับขี่ด้านซ้าย เช่น กรณีชาวต่างชาติ": "ไม่คุ้นเคยกับการขับขี่ด้านซ้าย",
            "เบรคกะทันหัน": "เบรกกะทันหัน",
            "เบรคกระทันหัน": "เบรกกะทันหัน",
            "เบครกระทันหัน": "เบรกกะทันหัน",
            "ถนนลื่น (เนื่องจากสภาพอากาศ)": "ถนนลื่น",
            "มีสิ่งรบกวนภายในรถ เช่น มีเด็ก มีสัตว์ภายในรถ": "มีสิ่งรบกวนภายในรถ",
            "แสงสว่างไม่เพียงพอหรือทัศนวิสัยกลางคืนไม่ดี": "แสงสว่างไม่เพียงพอ",
            "เข้าผิดช่องทางผิด": "เข้าผิดช่อง",
            "ไม่คุ้นชินกับยานพาหนะ เช่น กรณีเช่ารถ": "ไม่คุ้นเคยเส้นทาง/ขับรถไม่ชำนาญ",
            "อุปกรณ์ยานพาหนะบกพร่อง (ระบุ)": "อุปกรณ์ยานพาหนะบกพร่อง"
        })

    if 'เวลา' in df.columns:
        df['เวลา'] = df['เวลา'].apply(clean_time_str)
        df['ชั่วโมง'] = (
            df['เวลา']
            .astype(str)
            .str.extract(r'^(\d{1,2}):')[0]
        )
        df['ชั่วโมง'] = pd.to_numeric(
            df['ชั่วโมง'],
            errors='coerce'
        ).fillna(-1).astype(int)

    if 'วันที่เกิดเหตุ' in df.columns:

        df['วันที่เกิดเหตุ'] = df['วันที่เกิดเหตุ'].apply(parse_mixed_dates)
        df['year'] = df['วันที่เกิดเหตุ'].dt.year
        df['month'] = df['วันที่เกิดเหตุ'].dt.month
        df['day'] = df['วันที่เกิดเหตุ'].dt.day
        df['dayofweek'] = df['วันที่เกิดเหตุ'].dt.dayofweek
        df['is_weekend'] = (df['dayofweek'].isin([5, 6])).astype(int)
        df['is_new_year'] = df['วันที่เกิดเหตุ'].apply(
            lambda d: 1 if pd.notna(d) and
            ((d.month == 12 and d.day >= 29) or
             (d.month == 1 and d.day <= 4)) else 0
        )
        
        df['is_songkran'] = df['วันที่เกิดเหตุ'].apply(
            lambda d: 1 if pd.notna(d) and
            (d.month == 4 and 11 <= d.day <= 17) else 0
        )

        df['is_dangerous_7days'] = (
            (df['is_new_year'] == 1) |
            (df['is_songkran'] == 1)
        ).astype(int)
        
        df = add_day_type(df)

    if 'ชั่วโมง' in df.columns:

        df['time_period'] = df['ชั่วโมง'].apply(
            get_time_period
        )

    df['severity_level'] = df.apply(calculate_severity, axis=1)

    df = add_tambon(df, GEOJSON_PATH)

    drop_cols = [
        'ACC_CODE',
        'สายทางหน่วยงาน',
        'KM',
        'รหัสสายทาง',
        'วันที่เกิดเหตุ',
        'เวลา',
        'วันที่รายงาน',
        'เวลาที่รายงาน',
        'รถและคนที่เกิดเหตุ',
        'รถที่เกิดเหตุ',
        'dayofweek',
        'รถจักรยานยนต์',
        'รถสามล้อเครื่อง',
        'รถยนต์นั่งส่วนบุคคล',
        'รถตู้',
        'รถปิคอัพโดยสาร',
        'รถโดยสารมากกว่า4ล้อ',
        'รถปิคอัพบรรทุก4ล้อ',
        'รถบรรทุก6ล้อ',
        'รถบรรทุกไม่เกิน10ล้อ',
        'รถบรรทุกมากกว่า10ล้อ',
        'รถอีแต๋น',
        'รถอื่นๆ',
        'คนเดินเท้า',
        'บริเวณที่เกิดเหตุอื่น'
    ]

    df_cleaned = df.drop(
        columns=[
            col for col in drop_cols
            if col in df.columns
        ]
    )
    df_cleaned = df_cleaned.drop_duplicates()
    return df_cleaned

if __name__ == "__main__" and "ipykernel" not in sys.modules:

    if len(sys.argv) < 3:
        print("Error: กรุณาระบุ Path File Input และ Output ให้ครบถ้วน")
        sys.exit(1)

    input_path = sys.argv[1]
    output_path = sys.argv[2]

    if not os.path.exists(input_path):
        print(f"Error: ไม่พบไฟล์ตาม Path ที่ระบุ -> {input_path}")
        sys.exit(1)

    try:
        raw_df = pd.read_csv(
            input_path,
            encoding="utf-8-sig"
        )

        cleaned_df = process_cleansing_pipeline(raw_df)
        
        cleaned_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        print("Cleansing Successfully!")
        print("จำนวนแถวหลัง Cleaning:", len(cleaned_df))
        print("จำนวนคอลัมน์หลัง Cleaning:", len(cleaned_df.columns))

        print("จำนวนตำบลที่ไม่ระบุ:",(cleaned_df['tambon'] == 'ไม่ระบุ').sum())

    except Exception as e:

        print(f"Processing Error: {e}")
        sys.exit(1)

In [7]:
def run_cleaning(input_path, output_path):
    if not os.path.exists(input_path):
        print(f"Error: ไม่พบไฟล์ตาม Path ที่ระบุ -> {input_path}")
        return

    try:
        raw_df = pd.read_csv(input_path, encoding="utf-8-sig")

        cleaned_df = process_cleansing_pipeline(raw_df)

        cleaned_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        print("Cleansing Successfully!")
        print(f"ข้อมูลก่อน Cleaning: {len(raw_df):,} แถว")
        print(f"ข้อมูลหลัง Cleaning: {len(cleaned_df):,} แถว")
        print(f"ข้อมูลที่ถูกลบ: {len(raw_df) - len(cleaned_df):,} แถว")

    except Exception as e:
        print(f"Processing Error: {e}")

In [8]:
import pandas as pd

Q95 = 65.0
MIN_SCORE = 5.0

GROUP_KEYS = [
    "province_code",
    "district_code",
    "subdistrict_code",
    "time_period",
    "day_type",
    "year"
]

def calculate_risk_score(df):
    df = df.copy()

    for col in ["deaths", "serious_injuries", "minor_injuries"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["case_severity_score"] = (df["deaths"] * 25 + df["serious_injuries"] * 15 + df["minor_injuries"] * 5 + 5)
    df["total_injuries"] = (df["serious_injuries"] + df["minor_injuries"])

    result = (
        df.groupby(GROUP_KEYS, dropna=False)
        .agg(
            total_accidents=("year", "count"),
            total_deaths=("deaths", "sum"),
            total_injuries=("total_injuries", "sum"),
            total_severity_score=("case_severity_score", "sum")
        )
        .reset_index()
    )

    score = result["total_severity_score"].clip(upper=Q95)

    result["risk_score"] = (MIN_SCORE + ((score - MIN_SCORE) / (Q95 - MIN_SCORE)) * 95).round(2)

    result["risk_level"] = result["risk_score"].apply(
        lambda x: "high" if x >= 60
        else "medium" if x >= 30
        else "low"
    )

    return result.drop(columns=["total_severity_score"])

In [9]:
print(os.path.exists("accident2026.csv"))

True


In [10]:
input_path = "accident2026.csv"
output_path = "cleaned_accident_test.csv"

run_cleaning(input_path, output_path)

Cleansing Successfully!
ข้อมูลก่อน Cleaning: 15,239 แถว
ข้อมูลหลัง Cleaning: 15,052 แถว
ข้อมูลที่ถูกลบ: 187 แถว


In [11]:
df_test = pd.read_csv(
    "cleaned_accident_test.csv",
    encoding="utf-8-sig"
)

print("จำนวนแถว:", len(df_test))
print("จำนวนคอลัมน์:", len(df_test.columns))
print("ข้อมูลซ้ำ:", df_test.duplicated().sum())

display(df_test.head())

จำนวนแถว: 15052
จำนวนคอลัมน์: 27
ข้อมูลซ้ำ: 0


,ปีที่เกิดเหตุ,หน่วยงาน,สายทาง,จังหวัด,รถคันที่1,บริเวณที่เกิดเหตุ,มูลเหตุสันนิษฐาน,ลักษณะการเกิดเหตุ,สภาพอากาศ,LATITUDE,...,month,day,is_weekend,is_new_year,is_songkran,is_dangerous_7days,day_type,time_period,severity_level,tambon
0,2026,กรมทางหลวง,นิคมคำสร้อย - หนองสูง,มุกดาหาร,รถจักรยานยนต์,ทางตรง+ไม่มีความลาดชัน,ขับรถย้อนศร,ชนสิ่งกีดขวาง (บนผิวจราจร),แจ่มใส,16.381335,...,1,1,0,1,0,1,new_year,ดึก (00:00 - 03:59),60,นิคมคำสร้อย
1,2026,กรมทางหลวง,พญาเย็น - นครราชสีมา,นครราชสีมา,รถยนต์นั่งส่วนบุคคล/รถยนต์นั่งสาธารณะ,ไม่ระบุ,ขับรถเร็วเกินอัตรากำหนด,พลิกคว่ำ/ตกถนนในทางตรง,แจ่มใส,14.853079,...,1,1,0,1,0,1,new_year,ดึก (00:00 - 03:59),10,คลองไผ่
2,2026,กรมทางหลวง,บ้านโป่ง - พร้าว,เชียงใหม่,รถปิคอัพบรรทุก 4 ล้อ,ทางตรง+ไม่มีความลาดชัน,ขับรถเร็วเกินอัตรากำหนด,ชนในทิศทางตรงกันข้าม (ไม่ใช่การแซง),แจ่มใส,19.089352,...,1,1,0,1,0,1,new_year,ดึก (00:00 - 03:59),35,แม่หอพระ
3,2026,กรมทางหลวง,บ้านหว้า - อุทัย,อยุธยา,รถจักรยานยนต์,ไม่ระบุ,ฝ่าฝืนสัญญาณไฟ/เครื่องหมายจราจร,ชนเป็นมุมบริเวณทางแยก,แจ่มใส,14.324264,...,1,1,0,1,0,1,new_year,ดึก (00:00 - 03:59),20,คานหาม
4,2026,กรมทางหลวงชนบท,แยกทางหลวงหมายเลข 409 (กม.ที่ 7+204) - น้ำตกทร...,ปัตตานี,รถจักรยานยนต์,ทางตรง+ไม่มีความลาดชัน,ขับรถเร็วเกินอัตรากำหนด,ไม่ระบุ,แจ่มใส,6.676034,...,1,1,0,1,0,1,new_year,ดึก (00:00 - 03:59),10,นาประดู่
